# Visualization examples
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msoley/DSCI549/blob/master/In-class%20exercises/Practicum6/Visualization_examples.ipynb)

In this practicum we revisit the traffic counter from the KNIME practicum: one count of vehicles per day at a residential intersection, 2022–2024. A bar opened across the street on **1 April 2023**. Residents say it brought Friday/Saturday traffic; the bar owners say traffic was rising anyway.

We look at the same data with the basic chart types, and then leave the intersection for a world map:

1. **Histogram** – what values occur, and how often?
2. **Box, violin and strip plots** – comparing distributions across groups
3. **Time series** – how does traffic change over time?
4. **Bar chart** – comparing groups
5. **Map** – life expectancy around the world (Gapminder)

We use **seaborn**, a library built on top of matplotlib that makes good-looking statistical plots in one line, and **plotly** for the interactive map. The code is written for you. Your job is to **modify** it and **interpret** the result:
* **✏️ Modify** cells ask you to change something in the code cell right above and rerun it (`Shift+Enter`).
* **🤔 Think** cells ask a question. Write a short answer in the notebook.

*AI use disclaimer: Claude was used in creating this Practicum!*

## Setup

`sns.set_theme` sets a clean style for all plots; the `colorblind` palette stays distinguishable for people with color vision deficiency.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style='whitegrid', palette='colorblind', font_scale=1.1)

Load the raw data. The file has no proper header, so we name the columns ourselves.

In [ ]:
url = 'https://raw.githubusercontent.com/msoley/DSCI549/master/In-class%20exercises/Practicum5/traffic_counts_practicum.csv'
raw = pd.read_csv(url)
raw.columns = ['Date', 'Vehicles']
raw['Date'] = pd.to_datetime(raw['Date'], format='%m/%d/%y')
print(raw.shape)
raw.head()

## 1. Histogram
A histogram splits the range of values into **bins** and counts how many days fall into each bin. Let's look at the raw data before cleaning anything.

In [ ]:
ax = sns.histplot(data=raw, x='Vehicles', bins=30)
ax.set(xlabel='Vehicles per day', ylabel='Number of days', title='Distribution of daily traffic (raw data)')
plt.show()

> **🤔 Think:** There is a separate spike on the far left. What values are in it, and are they real traffic counts? Would you have noticed them by looking at `raw.head()` or `raw['Vehicles'].mean()`?

> **✏️ Modify:** Replace `bins=30` with `binwidth=1`. How many different values hide in that spike?

As in KNIME, we drop the `-1` sentinel values and the zero days. We also add the day of the week, mark Friday/Saturday (the bar's busy nights), the year and month, and whether the day is before or after the bar opened.

In [ ]:
traffic = raw[raw['Vehicles'] > 0].copy()
traffic['weekday'] = traffic['Date'].dt.day_name()
traffic['day_type'] = traffic['weekday'].isin(['Friday', 'Saturday']).map({True: 'Fri–Sat', False: 'Sun–Thu'})
traffic['year'] = traffic['Date'].dt.year
traffic['month'] = traffic['Date'].dt.month

BAR_OPENS = pd.Timestamp('2023-04-01')
traffic['period'] = (traffic['Date'] >= BAR_OPENS).map({False: 'Before the bar opened', True: 'After the bar opened'})
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
periods = ['Before the bar opened', 'After the bar opened']
print(len(raw) - len(traffic), 'days removed,', len(traffic), 'left')
traffic.head()

In [ ]:
ax = sns.histplot(data=traffic, x='Vehicles', bins=30)
ax.set(xlabel='Vehicles per day', ylabel='Number of days', title='Distribution of daily traffic')
plt.show()

> **✏️ Modify:** Change `bins=30` to `bins=5`, then `bins=100`. What do you lose with too few bins? What goes wrong with too many?

> **✏️ Modify:** Add `kde=True` to `sns.histplot(...)`. A smooth curve (a *kernel density estimate*) is drawn on top. Does it help or hide anything?

> **✏️ Modify:** Add `hue='day_type'` to split the histogram into Friday/Saturday and the other days.

> **🤔 Think:**
> 1. Which group has more days in total? Does that make it hard to compare the two shapes? Fix it by adding `stat='density', common_norm=False`. What does the y axis mean now? Update the y label.
> 2. Try `element='step'` and `multiple='stack'`. Which version makes the comparison easiest?

## 2. Box, violin and strip plots
A histogram shows one distribution well, but comparing seven of them (one per weekday) would need seven histograms. These plots put many distributions side by side:
* **Box plot** – five numbers: the box spans the middle 50% of the days (Q1 to Q3), the line inside is the median, the whiskers reach the most extreme values within 1.5 × the box height, and dots beyond that are outliers.
* **Violin plot** – the full shape of the distribution (a smoothed histogram, mirrored).
* **Strip plot** – every single day as one dot.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
sns.boxplot(data=traffic, x='weekday', y='Vehicles', order=days, ax=ax)
ax.set(xlabel='', ylabel='Vehicles per day', title='Daily traffic by weekday, 2022–2024')
plt.show()

> **🤔 Think:**
> 1. Which days have the highest median? Which day has the widest box (the most variable traffic)?
> 2. Friday and Saturday have the longest boxes and whiskers. Why could that be? (Hint: the data covers both before and after the bar opened.)

> **✏️ Modify:** Replace `sns.boxplot` with `sns.violinplot`. Does any day have a distribution with **two bumps**? Then try `sns.stripplot` and add `alpha=0.4` so overlapping dots stay visible. What does the strip plot show that the box plot cannot?

> **✏️ Modify:** Go back to `sns.violinplot` and add `hue='period', hue_order=periods, split=True, inner='quart'`. Each violin now has a *before* half and an *after* half. Which days changed shape after April 2023?

> **✏️ Modify:** Add `palette=['lightgray', 'tab:red']` to make "after" stand out, and rewrite the title so that it states your conclusion.

## 3. Time series
### 3.1 Daily counts with a rolling mean
The raw daily series is noisy. A **rolling mean** replaces each day with the average of the surrounding `window` days. The dashed line marks the day the bar opened: context that helps the reader.

In [ ]:
window = 30   # days

traffic['rolling'] = traffic['Vehicles'].rolling(window, center=True).mean()

fig, ax = plt.subplots(figsize=(12, 4))
sns.lineplot(data=traffic, x='Date', y='Vehicles', color='lightgray', linewidth=0.8, label='Daily count', ax=ax)
sns.lineplot(data=traffic, x='Date', y='rolling', color='black', linewidth=2, label=f'{window}-day rolling mean', ax=ax)
ax.axvline(BAR_OPENS, color='crimson', linestyle='--')
ax.text(BAR_OPENS, ax.get_ylim()[1], ' bar opens', color='crimson', va='top')
ax.set(xlabel='', ylabel='Vehicles per day', title='Daily traffic at the intersection, 2022–2024')
plt.show()

> **✏️ Modify:** Set `window = 7`, then `window = 90`. Which window shows the **seasons** best? Which one still shows the **weekly** ups and downs?

> **✏️ Modify:** Delete the two lines with `BAR_OPENS` (the `axvline` and the `text`). Without the marker, would a reader see anything special about April 2023?

> **🤔 Think:** Traffic jumps right after April 2023. Can you conclude from this chart that the bar caused it? What else happens every year in April? (Look at 2022 and 2024.)

### 3.2 Comparing years
To separate the bar from the seasons, we compare **the same months across years**: one line per year, x axis = month, and one panel per day type. `sns.relplot` does the grouping for us: it averages all days in each month and draws a shaded band around each line.

In [ ]:
g = sns.relplot(data=traffic, x='month', y='Vehicles', hue='year', col='day_type',
                kind='line', marker='o', palette='colorblind', height=4, aspect=1.4)
g.set_titles('{col_name}')
g.set_axis_labels('Month', 'Mean vehicles per day')
g.set(xticks=range(1, 13))
plt.show()

> **🤔 Think:** Back to the dispute.
> 1. In which panel does the 2023 line jump away from 2022, and in which month does the jump start?
> 2. In the other panel, how do the years compare?
> 3. So who is right, the residents or the bar owners? Could both be partly right?
> 4. What do the shaded bands show? Why are they wider in the Fri–Sat panel? (Hint: how many days per month go into each point?)

> **✏️ Modify:** Add `facet_kws={'sharey': False}` to `sns.relplot(...)`. Compare the y axes of the two panels. Why is this version misleading when the panels sit side by side?

> **✏️ Modify:** Add `errorbar=None` to remove the bands. Then swap the roles: `hue='day_type'` and `col='year'`. Which version answers the question better, and why?

> **✏️ Modify:** The colorblind palette makes the three years look like unrelated categories, but years have an order. Try `palette='Reds'` (light = earlier, dark = later), or highlight the comparison with `palette={2022: 'lightgray', 2023: 'tab:orange', 2024: 'tab:red'}`.

## 4. Bar chart
Bars compare a number across groups. Here: the mean traffic per weekday, before and after the bar opened.

In [ ]:
data = traffic.copy()

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=data, x='weekday', y='Vehicles', hue='period', order=days,
            hue_order=periods, palette=['lightgray', 'tab:red'], ax=ax)
ax.set(xlabel='', ylabel='Mean vehicles per day', title='Mean daily traffic by weekday')
ax.legend(title='')
plt.show()
data.groupby(['weekday', 'period'])['Vehicles'].mean().unstack().loc[days].round(1)

> **🤔 Think:** What are the black lines on top of the bars? (Hint: by default seaborn shows a 95% confidence interval of the mean.) Which differences do you trust, and which could be noise?

> **✏️ Modify:** Add `ax.set_ylim(40, 80)` before `plt.show()`. Look at Monday. How many times taller is the red bar than the gray one now? What is the real ratio (see the table)? Why must a bar chart start at zero?

> **✏️ Modify:** Make the bars horizontal by swapping `x='weekday'` and `y='Vehicles'` (and fix the labels). When are horizontal bars easier to read? (Hint: think of long category names.)

> **🤔 Think:** "Before" is January 2022–March 2023; "after" is April 2023–December 2024. Every weekday went up, not only Friday and Saturday. Is this a fair comparison? (Remember section 3.1.)

> **✏️ Modify:** Make it a fair comparison: compare April–December 2022 with April–December 2023 only. Replace `data = traffic.copy()` with
> ```python
> data = traffic[(traffic['month'] >= 4) & (traffic['year'] <= 2023)].copy()
> ```
> Which weekdays changed now? Update the title so that it tells the reader what to conclude.

## 5. Map: life expectancy around the world
Time to leave the intersection. Plotly ships with the **Gapminder** data used by Hans Rosling: life expectancy, GDP per capita and population for 142 countries, every 5 years from 1952 to 2007.

In [ ]:
gap = px.data.gapminder()
gap2007 = gap[gap['year'] == 2007]
gap2007.head()

A **choropleth map** colors each country by a value. Plotly finds the country borders from the 3-letter ISO codes in `iso_alpha`. The map is **interactive**: hover over a country, scroll to zoom, drag to pan.

In [ ]:
fig = px.choropleth(gap2007, locations='iso_alpha', color='lifeExp', hover_name='country',
                    color_continuous_scale='Viridis', labels={'lifeExp': 'Life expectancy (years)'},
                    title='Life expectancy at birth, 2007')
fig.show()

> **🤔 Think:** Which region stands out? Hover over the darkest countries. What do they have in common?

> **✏️ Modify:** Change `color_continuous_scale` to `'Jet'` (a rainbow scale), then to `'Reds'`. In the rainbow version, do you see sharp borders between groups of countries that are not really in the data? Which scale makes "higher = better" easiest to read?

> **✏️ Modify:** Add `projection='orthographic'` and drag the globe around. Then try `projection='mercator'`. Compare the size of Greenland and Africa (Africa is about 14 times larger).

> **✏️ Modify:** Animate it: replace `gap2007` with `gap`, add `animation_frame='year'`, add `range_color=[30, 85]`, and update the title. Press ▶. Find two countries whose color gets *worse* at some point. What happened there? Why do we need `range_color`? (Try the animation without it.)

> **✏️ Modify:** Color by wealth instead: `color='gdpPercap'`. Almost the whole map has the same color. Why? Try `color=np.log10(gap2007['gdpPercap'])` (add `import numpy as np` at the top of the cell). What does the log scale do?

> **🤔 Think:** On this map, Russia and Canada take up a huge share of the colored area, while Bangladesh (about 150 million people in 2007) is barely visible. Does the size of a country on the map tell you how many people the color describes? How would a **cartogram** (from the lecture) fix this?

## 6. Final challenge: fix this chart
The chart below breaks several of the rules from the lecture. **Work in pairs.**
1. List every problem you can find.
2. Fix them by editing the cell. Keep the bar chart, but change anything else.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
sns.barplot(data=traffic, x='year', y='Vehicles', hue='year', palette=['red', 'green', 'blue'], ax=ax)
ax.set_ylim(40, 58)
ax.tick_params(labelsize=6)
ax.grid(True, linewidth=2, color='black')
plt.show()

> **🤔 Think:** After your fix, is the yearly mean the right number to answer the residents vs. bar owners question? Which chart from today answers it best, and why?